# Integración de la etapa Spark de Felipe

Este notebook revisa el ETL, las tres consultas, la comparación CSV–Parquet y ML. El contrato está en `config/pipeline.yaml`; `scripts/run_pipeline.py` llama a los módulos de `spark/` mediante `run(spark, cfg)`. El paso de calidad se ejecuta desde ese pipeline. Las celdas de ETL y ML escriben en HDFS y requieren que el clúster y los datos estén disponibles.

## 1 · Cargar el YAML y el código de Spark

Carlos monta el repositorio en `/home/jovyan/work`. Buscamos la raíz desde la carpeta de trabajo del kernel y leemos el YAML que ya está en GitHub.

In [ ]:
import sys
from pathlib import Path

repo = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "config" / "pipeline.yaml").is_file()
     and (p / "spark" / "src" / "airline_stage.py").is_file()),
    None,
)
if repo is None:
    raise FileNotFoundError(
        "Ejecuta el notebook desde el repositorio actualizado, montado en Jupyter."
    )

import yaml
with (repo / "config" / "pipeline.yaml").open(encoding="utf-8") as stream:
    cfg = yaml.safe_load(stream)

sys.path.insert(0, str(repo / "spark" / "src"))
from airline_stage import (
    discover_raw_files, hdfs_zone, read_flights, read_airports,
    read_carriers, mark_aptitude, query_1, query_2, query_3,
)

print("Repositorio:", repo)
print("Raw vuelos:", hdfs_zone(cfg, "raw_flights"))
print("Raw auxiliares:", hdfs_zone(cfg, "raw_lookup"))
print("Curated:", hdfs_zone(cfg, "curated"))
print("Results:", hdfs_zone(cfg, "results"))


## 2 · Usar el clúster Spark de Carlos

La dirección del master viene del YAML. Si ya había una sesión local en el kernel, reinícialo antes de continuar.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Felipe - prueba de integracion")
    .master(cfg["spark"]["master"])
    .config("spark.hadoop.dfs.replication", str(cfg["hdfs"]["replication"]))
    .getOrCreate()
)
if spark.sparkContext.master != cfg["spark"]["master"]:
    raise RuntimeError(
        f"El kernel usa {spark.sparkContext.master}, pero el YAML indica "
        f"{cfg['spark']['master']}. Reinicia el kernel."
    )
print("Master Spark:", spark.sparkContext.master)
print("Paralelismo disponible:", spark.sparkContext.defaultParallelism)
print("Spark UI:", spark.sparkContext.uiWebUrl)


## 3 · Descubrir los archivos raw

El pipeline de Carlos ya carga 22 CSV en `zones.raw_flights` y las dos tablas auxiliares en `zones.raw_lookup`. Se comprueba la presencia de todos los años indicados en el YAML.

In [ ]:
raw_files = discover_raw_files(spark, hdfs_zone(cfg, "raw_flights"))
expected_years = set(range(cfg["data"]["first_year"], cfg["data"]["last_year"] + 1))
missing_years = sorted(expected_years - raw_files.flights.keys())
if missing_years:
    raise FileNotFoundError(f"Faltan CSV anuales en raw: {missing_years}")

lookup_dir = hdfs_zone(cfg, "raw_lookup").rstrip("/")
lookup_names = set(cfg["data"]["lookup_files"])
if not {"airports.csv", "carriers.csv"} <= lookup_names:
    raise ValueError("data.lookup_files debe incluir airports.csv y carriers.csv")
airports = read_airports(spark, lookup_dir + "/airports.csv")
carriers = read_carriers(spark, lookup_dir + "/carriers.csv")
airports.createOrReplaceTempView("airports")
carriers.createOrReplaceTempView("carriers")
print("Años encontrados:", sorted(raw_files.flights))
print("Tablas auxiliares registradas: airports, carriers")


## 4 · Probar el ETL con 2001 y 2002

Estos son los años con bytes no UTF-8 en `TailNum`. La etapa Spark usa los campos necesarios y conserva las marcas y motivos de aptitud. Cambia `ANIOS_PRUEBA` a `None` para construir el plan con los 22 años; cada `show()` lanza un trabajo Spark.

In [ ]:
ANIOS_PRUEBA = [2001, 2002]
flights = mark_aptitude(read_flights(spark, raw_files, ANIOS_PRUEBA))
flights.createOrReplaceTempView("flights")
flights.select(
    "Year", "_anio_fuente", "Cancelled", "Diverted", "ArrDelay",
    "vuelo_valido", "apto_consulta_1", "apto_consulta_2",
    "apto_consulta_3", "apto_modelo", "motivo_vuelo",
).show(10, truncate=False)


## 5 · Consulta 1: media por aerolínea y año

El mínimo de vuelos por grupo se lee del YAML.

In [ ]:
minimum = int(cfg["min_flights_per_group"])
resultado_1 = query_1(flights, minimum)
resultado_1.show(20, truncate=False)


## 6 · Consulta 2: tres rutas por origen

Las medias negativas son válidas: indican llegada anticipada en promedio. Se ordena de mayor a menor media dentro de cada aeropuerto.

In [ ]:
resultado_2 = query_2(flights, minimum)
resultado_2.show(30, truncate=False)


## 7 · Consulta 3: aerolínea frente al origen

El promedio del origen se calcula con la suma de todos sus retrasos dividida por todos sus vuelos, antes de ocultar grupos de aerolínea con menos del mínimo.

In [ ]:
resultado_3 = query_3(flights, minimum)
resultado_3.show(20, truncate=False)


## 8 · Crear la zona curated en Parquet

Esta celda ejecuta la etapa ETL completa: lee los 22 CSV originales de HDFS, aplica las marcas de aptitud y escribe Parquet particionado por Year en la ruta del YAML. La primera ejecución puede tardar; repetirla reemplaza curated sin duplicar filas. Ejecútala cuando el clúster esté libre.


In [ ]:
import sys

ruta_spark = str(repo / "spark")
if ruta_spark not in sys.path:
    sys.path.insert(0, ruta_spark)
from etl import run as ejecutar_etl

resumen_etl = ejecutar_etl(spark, cfg)
print("ETL completado:", resumen_etl)


## 9 · Comparar el espacio ocupado por CSV y Parquet

Usamos 2007 en ambos formatos. Se mide el tamaño lógico de HDFS y el espacio físico consumido con la replicación. La carpeta Parquet corresponde únicamente a la partición Year=2007.


In [ ]:
anio_comparacion = 2007
ruta_csv = raw_files.flights[anio_comparacion]
ruta_parquet = hdfs_zone(cfg, "curated")
ruta_particion = (
    f"{ruta_parquet}/{cfg['partition_column']}={anio_comparacion}"
)

def tamanos_hdfs(uri):
    path = spark._jvm.org.apache.hadoop.fs.Path(uri)
    fs = path.getFileSystem(spark._jsc.hadoopConfiguration())
    if not fs.exists(path):
        raise FileNotFoundError(uri)
    resumen = fs.getContentSummary(path)
    return int(resumen.getLength()), int(resumen.getSpaceConsumed())

tam_csv = tamanos_hdfs(ruta_csv)
tam_parquet = tamanos_hdfs(ruta_particion)
for nombre, (logico, fisico) in (
    ("CSV original", tam_csv),
    ("Parquet curated", tam_parquet),
):
    print(
        f"{nombre}: {logico / 1024**2:.1f} MiB lógicos; "
        f"{fisico / 1024**2:.1f} MiB con replicación"
    )


## 10 · Medir el tiempo de lectura con la misma consulta

La optimización elegida es materializar los datos una vez como Parquet particionado por Year. Comparamos la misma consulta 1 sobre 2007 desde el CSV original y desde curated; repetimos en orden inverso y comprobamos que ambas devuelvan los mismos grupos y medias. Spark ejecuta la lectura al pedir el resultado, por eso cronometramos collect(), no read(). Cada ejecución tiene una etiqueta visible en Spark UI. Las medidas finales se harán en el portátil de referencia.


In [ ]:
import json
from statistics import median
from time import perf_counter
from pyspark.sql import functions as F

desde_csv = mark_aptitude(
    read_flights(spark, raw_files, [anio_comparacion])
)
desde_parquet = (
    spark.read.parquet(ruta_parquet)
    .filter(F.col(cfg["partition_column"]) == anio_comparacion)
)
minimo_grupo = int(cfg["min_flights_per_group"])
consulta_csv = query_1(desde_csv, minimo_grupo)
consulta_parquet = query_1(desde_parquet, minimo_grupo)

tiempos = {"CSV": [], "Parquet": []}
referencia = None
for intento, (nombre, consulta) in enumerate((
    ("CSV", consulta_csv),
    ("Parquet", consulta_parquet),
    ("Parquet", consulta_parquet),
    ("CSV", consulta_csv),
), start=1):
    grupo = f"consulta1_2007_{nombre.lower()}_{intento}"
    spark.sparkContext.setJobGroup(grupo, grupo)
    try:
        inicio = perf_counter()
        filas = consulta.collect()
        segundos = perf_counter() - inicio
    finally:
        spark.sparkContext.clearJobGroup()
    firma = sorted(
        (fila["Year"], fila["UniqueCarrier"], fila["n_vuelos"],
         round(fila["retraso_medio"], 6))
        for fila in filas
    )
    if referencia is None:
        referencia = firma
    elif firma != referencia:
        raise AssertionError(
            f"{nombre} no produjo los mismos resultados que CSV"
        )
    tiempos[nombre].append(segundos)
    print(f"{nombre}: {segundos:.2f} s; {len(filas)} grupos")

for nombre in ("CSV", "Parquet"):
    print(f"Mediana {nombre}: {median(tiempos[nombre]):.2f} s")
print("Resultados equivalentes:", True)

carpeta_evidencia = repo / "spark" / "output" / "benchmark_2007"
carpeta_evidencia.mkdir(parents=True, exist_ok=True)
resumen_benchmark = {
    "year": anio_comparacion,
    "query": "consulta_1",
    "csv_seconds": tiempos["CSV"],
    "parquet_seconds": tiempos["Parquet"],
    "csv_median_seconds": median(tiempos["CSV"]),
    "parquet_median_seconds": median(tiempos["Parquet"]),
    "csv_logical_bytes": tam_csv[0],
    "parquet_logical_bytes": tam_parquet[0],
    "csv_physical_bytes": tam_csv[1],
    "parquet_physical_bytes": tam_parquet[1],
    "results_equal": True,
}
(carpeta_evidencia / "comparacion.json").write_text(
    json.dumps(resumen_benchmark, indent=2), encoding="utf-8"
)
print("Medidas guardadas en:", carpeta_evidencia / "comparacion.json")


## 11 · Ver los planes de ejecución

Los planes muestran FileScan CSV frente a FileScan Parquet y los nodos Exchange que provocan shuffle. La lectura Parquet debería mostrar PartitionFilters sobre Year=2007. Guardamos ambos planes para el informe. Abre la Spark UI indicada en la celda 4; el Compose publica el puerto 4040. Si Spark eligió 4041, cierra los otros kernels Spark y reinicia este antes de medir. En Jobs busca consulta1_2007_*. Para cada formato, entra en Stages, apunta IDs y Shuffle Read/Write y haz capturas mientras este kernel siga abierto. Registra solo lo que muestre la ejecución; Parquet no será siempre más rápido.


In [ ]:
from contextlib import redirect_stdout
from io import StringIO

for nombre, consulta in (("csv", consulta_csv), ("parquet", consulta_parquet)):
    captura = StringIO()
    with redirect_stdout(captura):
        consulta.explain("formatted")
    plan = captura.getvalue()
    destino = carpeta_evidencia / f"plan_{nombre}.txt"
    destino.write_text(plan, encoding="utf-8")
    print(f"Plan {nombre} guardado en {destino}")
    print(plan)


## 12 · Integrar con el pipeline y documentar la comparación

Desde la raíz del repositorio, Carlos puede ejecutar el pipeline con el clúster encendido:

    docker compose exec jupyter python work/scripts/run_pipeline.py --steps etl quality analysis ml

El tiempo medido incluye lectura, filtros y agrupación de la consulta 1. Es una comparación del flujo real, no un cronómetro de la llamada a read, que es perezosa. El CSV conserva las 29 columnas originales; curated guarda las columnas necesarias y las marcas, de modo que la diferencia de tamaño incluye selección de columnas además del formato. Anota el equipo utilizado y adjunta capturas de la Spark UI al informe.


## 13 · Línea base de MLlib

El modelo predice ArrDelay > 15 sin usar información posterior al vuelo. Entrena con una muestra reproducible de 2006 y evalúa en 2007; evitamos 2008 porque el archivo publicado cubre solo parte del año. Usa mes, día de la semana, hora programada, aerolínea, origen, destino y distancia. Las categorías se codifican y una regresión logística produce AUC ROC y accuracy. La función guarda las métricas en results. Ejecuta esta celda después de crear curated.


In [ ]:
from ml_baseline import run as ejecutar_ml
resumen_ml = ejecutar_ml(spark, cfg)
print(resumen_ml)


Para el informe, anota el número de filas de entrenamiento y prueba, la proporción de retrasados y el AUC ROC. Es un baseline con dos años y muestreo del 2 %, no un modelo ajustado para producción. La validación temporal evita entrenar con vuelos posteriores a los usados para evaluar.
